# 이미지 CPU 분류: 픽셀·색·윤곽 특징

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

가중치 다운로드 없는 CPU 기준 모델입니다. 크기 정규화가 물체 비율을 왜곡할 수 있으므로 과제에 따라 패딩/크롭 정책을 변경하세요.

In [ ]:
DEMO=True
TRAIN_MANIFEST='data/train_images.csv';TEST_MANIFEST='data/test_images.csv'
IMAGE_ROOT='data';PATH_COL='path';ID='id';TARGET='label'
GROUP=None # 동일 사람·영상·촬영 세션 열이 있으면 필수 지정
FEATURE='pixels' # pixels / color_hist / gradient
IMAGE_SIZE=(24,24);METRIC='f1_macro'
PREDICTION_KIND='label';CLASS_ORDER=None;POSITIVE_CLASS=None
TARGET_COLUMNS=['label'];SAMPLE_PATH=None;OUTPUT='outputs/image/classification.csv'


## 공통 함수

학습·제출 함수

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 데이터

테스트 파일 순서와 ID를 보존합니다.

In [ ]:
from PIL import Image, ImageOps, ImageDraw
import hashlib

def open_rgb(path):
    with Image.open(path) as source:
        image=ImageOps.exif_transpose(source).convert('RGB')
        return np.asarray(image).copy()
if DEMO:
    directory=Path('outputs/demo_images');directory.mkdir(parents=True,exist_ok=True)
    rows=[]
    for i in range(80):
        arr=rng.integers(0,25,(32,32,3),dtype=np.uint8);label=i%2
        arr[8:24,8:24,label]=rng.integers(180,255,(16,16),dtype=np.uint8)
        p=directory/f'image_{i}.png';Image.fromarray(arr).save(p)
        rows.append({ID:i,PATH_COL:str(p.resolve()),TARGET:label})
    train=pd.DataFrame(rows[:64]);test=pd.DataFrame(rows[64:]).drop(columns=TARGET)
else:
    train=read_table(TRAIN_MANIFEST);test=read_table(TEST_MANIFEST)
    for frame in (train,test): frame[PATH_COL]=frame[PATH_COL].map(lambda p:str(Path(IMAGE_ROOT)/str(p)))
check_ids(train,ID);check_ids(test,ID)
if train[TARGET].isna().any(): raise ValueError('이미지 정답 결측')
# 손상된 이미지를 조용히 버리면 제출 행이 어긋나므로 로딩 오류를 그대로 알립니다.
train_images=[open_rgb(p) for p in train[PATH_COL]]
test_images=[open_rgb(p) for p in test[PATH_COL]]


## 전처리·특징 선택

라벨 정보를 사용하지 않는 이미지별 특징 계산 후 학습 분할로 표준화를 학습합니다.

In [ ]:
def image_features(images,method=FEATURE):
    vectors=[]
    for arr in images:
        small=np.asarray(Image.fromarray(arr).resize(IMAGE_SIZE),dtype=float)/255
        if method=='pixels': vector=small.ravel()
        elif method=='color_hist': vector=np.concatenate([np.histogram(small[:,:,k],bins=16,range=(0,1),density=False)[0]/small.shape[0]/small.shape[1] for k in range(3)])
        elif method=='gradient':
            gray=small.mean(axis=2);gy,gx=np.gradient(gray)
            mag=np.hypot(gx,gy);angle=np.arctan2(gy,gx)
            vectors_grid=[]
            for ys in np.array_split(np.arange(len(gray)),4):
                for xs in np.array_split(np.arange(gray.shape[1]),4):
                    ix=np.ix_(ys,xs);vectors_grid.extend(np.histogram(angle[ix],bins=9,range=(-np.pi,np.pi),weights=mag[ix])[0])
            vector=np.asarray(vectors_grid);vector=vector/(np.linalg.norm(vector)+1e-8)
        else: raise ValueError(method)
        vectors.append(vector)
    return np.asarray(vectors)


## 분류 학습·평가

파일 중복은 해시로 묶지만 근접 영상 프레임은 해시로 잡히지 않습니다. GROUP을 촬영 세션으로 지정하세요.

In [ ]:
X=image_features(train_images);Xt=image_features(test_images);y=train[TARGET];TASK='classification'
if GROUP:
    a,b=split_rows(train,TARGET,TASK,'group',GROUP)
else:
    # 동일 이미지 중복도 다른 분할로 보내지 않도록 해시를 그룹으로 사용
    train['__hash__']=[hashlib.sha256(a.tobytes()+str(a.shape).encode()).hexdigest() for a in train_images]
    a,b=split_rows(train,TARGET,TASK,'group','__hash__')
candidates={'dummy':make_pipeline(StandardScaler(),DummyClassifier(strategy='prior')),
    'linear':make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced',random_state=SEED)),
    'trees':ExtraTreesClassifier(n_estimators=80,min_samples_leaf=2,random_state=SEED,n_jobs=1)}
best,validation_model,results=fit_compare(candidates,X,y,a,b,TASK,METRIC)
print(classification_report(y.iloc[b],validation_model.predict(X[b]),zero_division=0))


## 재학습·제출

선택한 CPU 모델로 전체 학습과 예측을 수행합니다.

In [ ]:
final_model=clone(candidates[best]).fit(X,y)
if TASK=='classification': predictions=classification_output(final_model,Xt,PREDICTION_KIND,CLASS_ORDER,POSITIVE_CLASS)
else: predictions=final_model.predict(Xt)
submission=write_submission(test[ID],predictions,ID,TARGET_COLUMNS,OUTPUT,SAMPLE_PATH,TASK=='classification' and PREDICTION_KIND!='label')
